# 08 · T grid 정교화 — sub-lever 단일축 fine sweep (EXP #35, LB 0.6916)

직전 = #33 Soft+T=1.5(LB 0.6914). T=1.5는 #33 grid {0.7,1.0,1.5}의 **경계 최댓값**이라 진짜 peak가 그 너머인지 미확인. (노트북 생성 라벨은 e34이나 실험 번호는 #35.)

**가정**: #33에서 T축이 sub-lever space의 dominant 축으로 확정됐으니(max_w·α dead), T만 9-point fine grid로 넓히면 T=1.5 너머에 local maximum이 있을 수 있다. T<1 sharpen은 약한 classifier에 noise injection일 것.

**실험**: T ∈ {0.5~3.0} 9-point을 Δ=∞·α=1.0 고정, cache 재사용(학습 0). Floor=#33 best 초과 strict로 OOF overfit 차단 + T=1.5 재선택 차단.

**결과**: **Δo span 0.0003 ≪ noise → plateau 확정**(T축 ceiling). best=T=0.5(+0.0001 vs T=1.5)가 경계에 위치 → T=0.5 채택 **LB 0.6916 (+0.0002, sub-noise)**. T 정교화 라인 종료 — 같은 라인 미세조정의 한계, 다음은 구조적 새 신호가 필요하다는 신호.

## 셀 0 — imports + 상수 + T grid 정의

In [ ]:
import os, time, json, itertools
import numpy as np
import pandas as pd

DT, DT_PRED = 0.04, 0.08
N_FOLDS = 5
MINORITY_THRESHOLD = 5.0
HIT_RADIUS = 0.01
SEEDS = [42, 7, 123]

# K15/GRU reference HR (#32 셀 5/6 정량)
K15_REF_HR = dict(overall=0.6649, major=0.7312, minor=0.3105)
GRU_REF_OVERALL = 0.6659
GRU_REF_MINOR = 0.3181
ORACLE_KG_HR = dict(overall=0.6870, major=0.7474, minor=0.3638)
CEILING_GAIN_KG = dict(overall=0.0221, major=0.0162, minor=0.0533)

# Soft baseline (#32, T=1.0) + #33 best (T=1.5) 실측 OOF
SOFT_BASELINE_HR_OVERALL = 0.6686    # T=1.0 (#32 Soft-base)
SOFT_BASELINE_DO = 0.0037            # T=1.0 Δo (참고)
E33_BEST_HR_OVERALL = 0.6687         # T=1.5 (#33 채택, LB 0.6914)
E33_BEST_DO = 0.0038                 # T=1.5 Δo → Floor primary 임계

# T grid (1-D fine sweep, plan §4.1). Δ=∞, α=1.0 고정.
T_GRID = [0.5, 0.7, 1.0, 1.2, 1.5, 1.8, 2.0, 2.5, 3.0]
FIXED_DELTA = float('inf')
FIXED_ALPHA = 1.0
BASELINE_CELL = (1.0, float('inf'), 1.0)   # Soft-base (#32)
E33_BEST_CELL = (1.5, float('inf'), 1.0)   # #33 채택 cell
N_CELLS = len(T_GRID)
assert N_CELLS == 9

# Phase A 가드 — G1 informational + Floor primary (= #33 best 초과)
COMBINED_STD = 0.0014
G1_INFO_THR  = COMBINED_STD * (N_CELLS ** 0.5)   # ≈ 0.0042 informational only
G4_THR       = COMBINED_STD * (2 ** 0.5)         # ≈ 0.0020 primary per-cell
FLOOR_PRIMARY = E33_BEST_DO                       # ≈ 0.0038 primary (#33 best 초과)

os.makedirs('results', exist_ok=True)
print(f'SEEDS={SEEDS}')
print(f'T grid (1-D): {T_GRID}  (Δ={FIXED_DELTA}, α={FIXED_ALPHA} 고정) = {N_CELLS} cell')
print(f'Baseline cell (#32 Soft-base): T={BASELINE_CELL[0]}  |  #33 best cell: T={E33_BEST_CELL[0]}')
print(f'G1 informational: +{G1_INFO_THR:.4f} (√{N_CELLS}·{COMBINED_STD:.4f})')
print(f'G4 primary: +{G4_THR:.4f} (per-cell LB자격)')
print(f'Floor primary: +{FLOOR_PRIMARY:.4f} (= #33 best T=1.5 Δo 초과, OOF overfit 차단)')
print(f'K15 ref overall={K15_REF_HR["overall"]:.4f} | Soft base OOF={SOFT_BASELINE_HR_OVERALL:.4f} (+{SOFT_BASELINE_DO:.4f}) | #33 best OOF={E33_BEST_HR_OVERALL:.4f} (+{E33_BEST_DO:.4f})')


## 셀 1 — helpers + `apply_soft_lever` (T temperature 변환, #33 비트 동일)

In [ ]:
def _norm(x):
    return np.linalg.norm(x, axis=-1)


def physics_baseline(traj):
    p0, p_m40 = traj[:, -1, :], traj[:, -2, :]
    v_last = (p0 - p_m40) / DT
    return (p0 + v_last * DT_PRED).astype(np.float32)


def hit_rate(pred, label, r=HIT_RADIUS):
    return float((np.linalg.norm(pred - label, axis=1) < r).mean())


def hr_triple(hit_arr, minority_mask):
    return dict(
        overall=float(hit_arr.mean()),
        major=float(hit_arr[~minority_mask].mean()),
        minor=float(hit_arr[minority_mask].mean()),
    )


def apply_soft_lever(p, T=1.0, Delta=float('inf'), alpha=1.0):
    """sub-lever 변환 (#33 비트 동일): p → σ scaling → max_w clip → centered shrinkage → w.

    1. σ (temperature T): w_T = sigmoid(logit(p) / T)  [T==1.0 → w_T = p]
    2. max_w (cap Δ): w_D = clip(w_T, 0.5−Δ, 0.5+Δ)  [Δ==inf → w_D = w_T]
    3. centered shrinkage α: w' = 0.5 + α (w_D − 0.5)  [α==1.0 → w' = w_D]

    EXP #34 는 항상 Δ=∞, α=1.0 → 사실상 temperature 변환만. baseline (T=1) → w'=p identity.
    """
    if T == 1.0:
        w = p.astype(np.float32).copy()
    else:
        eps = 1e-6
        p_clip = np.clip(p, eps, 1 - eps)
        logit = np.log(p_clip / (1 - p_clip))
        w = (1.0 / (1.0 + np.exp(-logit / T))).astype(np.float32)
    if Delta < float('inf'):
        w = np.clip(w, 0.5 - Delta, 0.5 + Delta)
    if alpha != 1.0:
        w = 0.5 + alpha * (w - 0.5)
    return w.astype(np.float32)


def cell_name(T, Delta, alpha):
    """Cell 명명 encoding (#33 동일): T{T*10}_D{Δ*100|INF}_a{α*100:03d}."""
    t_s = f'T{int(round(T*10)):02d}'
    d_s = 'DINF' if Delta == float('inf') else f'D{int(round(Delta*100)):02d}'
    a_s = f'a{int(round(alpha*100)):03d}'
    return f'{t_s}_{d_s}_{a_s}'


# Sanity: baseline & #33-best cell apply_soft_lever
_p_test = np.array([0.1, 0.3, 0.5, 0.7, 0.9], dtype=np.float32)
_w_base = apply_soft_lever(_p_test, *BASELINE_CELL)
assert np.allclose(_w_base, _p_test, atol=1e-6), f'baseline cell non-identity: {_w_base} vs {_p_test}'
print('apply_soft_lever baseline (T=1) identity OK')
print(f'  baseline cell name: {cell_name(*BASELINE_CELL)}  |  #33 best cell name: {cell_name(*E33_BEST_CELL)}')

# Print all 9 cell names
print(f'\n9 cell names (Δ=∞, α=1.0 고정):')
for i, T in enumerate(T_GRID):
    tag = ''
    if (T, FIXED_DELTA, FIXED_ALPHA) == BASELINE_CELL: tag = ' (#32 Soft-base)'
    if (T, FIXED_DELTA, FIXED_ALPHA) == E33_BEST_CELL: tag = ' (#33 채택 best)'
    print(f'  [{i+1}] {cell_name(T, FIXED_DELTA, FIXED_ALPHA):<18}  T={T}{tag}')


## 셀 2 — Drive mount + 데이터 + cache 동기화

**필수 cache 18개** (#33 과 동일):
- K15 OOF/test (6): EXP #29 `topk_e29_{oof,test}_K15_seed*.npy`
- GRU OOF/test (6): EXP #25 `gru_e25_fren_{oof,test}_seed*.npy`
- **#32 classifier OOF/test (6)**: `split_e32_clf_{oof,test}_seed*.npy` — Phase A 학습 0 의 핵심

cache 누락 시 lever 폐기 (재학습 안 함).


In [ ]:
CACHE_TRAJ_TR = 'traj_train.npy'
CACHE_Y_TR    = 'y_train.npy'
CACHE_TRAJ_TS = 'traj_test.npy'

K15_OOF_PATHS    = [f'results/topk_e29_oof_K15_seed{s}.npy' for s in SEEDS]
K15_TEST_PATHS   = [f'results/topk_e29_test_K15_seed{s}.npy' for s in SEEDS]
GRU_OOF_PATHS    = [f'results/gru_e25_fren_oof_seed{s}.npy' for s in SEEDS]
GRU_TEST_PATHS   = [f'results/gru_e25_fren_test_seed{s}.npy' for s in SEEDS]
CLF32_OOF_PATHS  = [f'results/split_e32_clf_oof_seed{s}.npy' for s in SEEDS]
CLF32_TEST_PATHS = [f'results/split_e32_clf_test_seed{s}.npy' for s in SEEDS]
all_required = (K15_OOF_PATHS + K15_TEST_PATHS + GRU_OOF_PATHS + GRU_TEST_PATHS
                + CLF32_OOF_PATHS + CLF32_TEST_PATHS)
assert len(all_required) == 18

need_mount = (
    not (os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR))
    or any(not os.path.exists(p) for p in all_required)
)

if need_mount:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/open.zip'
    if not os.path.exists('/content/open'):
        !unzip -q -o "{ZIP_PATH}" -d /content/

for p in all_required:
    if not os.path.exists(p):
        drive_p = f'/content/drive/MyDrive/{p}'
        if os.path.exists(drive_p):
            !cp {drive_p} {p}
missing = [p for p in all_required if not os.path.exists(p)]
assert not missing, f'필수 cache 누락 (K15/GRU/#32 classifier): {missing}'
print(f'K15 + GRU + #32 classifier cache 18/18 OK')


def _resolve_data_dir():
    for cand in ['/content/open', '/content']:
        if os.path.exists(f'{cand}/train_labels.csv'):
            return cand
    return None

DATA_DIR = _resolve_data_dir()


def _resolve_sample_sub(data_dir):
    for p in [f'{data_dir}/sample_submission.csv', f'{data_dir}/submission/sample_submission.csv']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'sample_submission.csv not found under {data_dir}')


if os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR):
    traj_train = np.load(CACHE_TRAJ_TR)
    y_train    = np.load(CACHE_Y_TR)
else:
    labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
    train_ids = labels['id'].tolist()
    y_train = labels[['x','y','z']].values.astype(np.float32)
    traj_train = np.empty((len(train_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(train_ids):
        df = pd.read_csv(f'{DATA_DIR}/train/{tid}.csv')
        traj_train[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TR, traj_train)
    np.save(CACHE_Y_TR, y_train)

sample_sub_path = _resolve_sample_sub(DATA_DIR)
sample_sub = pd.read_csv(sample_sub_path)
test_ids = sample_sub['id'].tolist()
if os.path.exists(CACHE_TRAJ_TS):
    traj_test = np.load(CACHE_TRAJ_TS)
else:
    traj_test = np.empty((len(test_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(test_ids):
        df = pd.read_csv(f'{DATA_DIR}/test/{tid}.csv')
        traj_test[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TS, traj_test)

assert traj_train.shape == (10000, 11, 3) and traj_test.shape == (10000, 11, 3)
print(f'train {traj_train.shape}, test {traj_test.shape}, cache 18/18 OK')


## 셀 3 — base + Frenet frame + minority mask (#33 비트 동일)

In [ ]:
def build_frenet_batch(v_last, a_sm, fb_thresh=1e-6):
    vn = np.linalg.norm(v_last, axis=1, keepdims=True)
    eL = v_last / (vn + 1e-9)
    a_perp = a_sm - (a_sm * eL).sum(1, keepdims=True) * eL
    apn = np.linalg.norm(a_perp, axis=1, keepdims=True)
    eN_p = a_perp / (apn + 1e-9)
    z_up = np.array([0., 0., 1.]); y_up = np.array([0., 1., 0.])
    n1 = np.linalg.norm(np.cross(eL, z_up), axis=1, keepdims=True)
    eN_fb1 = np.cross(eL, z_up) / (n1 + 1e-9)
    eN_fb2 = np.cross(eL, y_up); n2 = np.linalg.norm(eN_fb2, axis=1, keepdims=True)
    eN_fb  = np.where(n1 < 1e-6, eN_fb2 / (n2 + 1e-9), eN_fb1)
    eN = np.where(apn < fb_thresh, eN_fb, eN_p)
    eB = np.cross(eL, eN)
    eB = eB / (np.linalg.norm(eB, axis=1, keepdims=True) + 1e-9)
    return eL.astype(np.float32), eN.astype(np.float32), eB.astype(np.float32)


def compute_kinematics(traj):
    v = (traj[:, 1:, :] - traj[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last = v[:, -1, :]
    a_last = a[:, -1, :]
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    w = np.array([1, 2, 3]) / 6
    a_sm = (a[:, -3:, :] * w[None, :, None]).sum(axis=1)
    return v_last.astype(np.float32), a_last.astype(np.float32), jerk_last.astype(np.float32), a_sm.astype(np.float32)


base_train = physics_baseline(traj_train)
base_test  = physics_baseline(traj_test)
vl_tr, al_tr, jl_tr, asm_tr = compute_kinematics(traj_train)
vl_ts, al_ts, jl_ts, asm_ts = compute_kinematics(traj_test)
eL_tr, eN_tr, eB_tr = build_frenet_batch(vl_tr, asm_tr)
eL_ts, eN_ts, eB_ts = build_frenet_batch(vl_ts, asm_ts)

acc_norm_last_tr = _norm(al_tr)
acc_norm_last_ts = _norm(al_ts)
minority_mask_tr = acc_norm_last_tr >= MINORITY_THRESHOLD
minority_mask_ts = acc_norm_last_ts >= MINORITY_THRESHOLD
print(f'minority train {minority_mask_tr.sum()}/{len(minority_mask_tr)} ({minority_mask_tr.mean():.4f})')
print(f'minority test  {minority_mask_ts.sum()}/{len(minority_mask_ts)} ({minority_mask_ts.mean():.4f})')


## 셀 4 — K15 + GRU cache 로드 + HR sanity + GRU form auto-detect (#32/#33 셀 5 동일)

In [ ]:
def load_seed_avg(paths):
    return np.mean([np.load(p) for p in paths], axis=0).astype(np.float32)


oof_K15 = load_seed_avg(K15_OOF_PATHS)
test_K15 = load_seed_avg(K15_TEST_PATHS)
assert oof_K15.shape == (10000, 3) and test_K15.shape == (10000, 3)

pred_K15_tr = base_train + oof_K15
pred_K15_ts = base_test + test_K15
err_K15 = np.linalg.norm(y_train - pred_K15_tr, axis=1)
hit_K15 = err_K15 < HIT_RADIUS
HR_K15 = hr_triple(hit_K15, minority_mask_tr)
print(f'K15 HR: overall={HR_K15["overall"]:.4f}  major={HR_K15["major"]:.4f}  minor={HR_K15["minor"]:.4f}')

for k in ['overall', 'major', 'minor']:
    diff = abs(HR_K15[k] - K15_REF_HR[k])
    assert diff < 0.0005, f'K15 {k} drift: ref {K15_REF_HR[k]:.4f} vs got {HR_K15[k]:.4f}'
print('K15 sanity OK')

oof_GRU = load_seed_avg(GRU_OOF_PATHS)
test_GRU = load_seed_avg(GRU_TEST_PATHS)

pred_cart = base_train + oof_GRU
err_cart = np.linalg.norm(y_train - pred_cart, axis=1)
HR_cart_overall = float((err_cart < HIT_RADIUS).mean())

oof_GRU_inv = (oof_GRU[:, 0:1] * eL_tr + oof_GRU[:, 1:2] * eN_tr + oof_GRU[:, 2:3] * eB_tr)
pred_fren = base_train + oof_GRU_inv
err_fren = np.linalg.norm(y_train - pred_fren, axis=1)
HR_fren_overall = float((err_fren < HIT_RADIUS).mean())

print(f'\nGRU form auto-detect:')
print(f'  Cart    HR = {HR_cart_overall:.4f}  (|Δ vs {GRU_REF_OVERALL}| = {abs(HR_cart_overall - GRU_REF_OVERALL):.4f})')
print(f'  Frenet  HR = {HR_fren_overall:.4f}  (|Δ vs {GRU_REF_OVERALL}| = {abs(HR_fren_overall - GRU_REF_OVERALL):.4f})')

if abs(HR_cart_overall - GRU_REF_OVERALL) < 0.005:
    pred_GRU_tr = pred_cart
    pred_GRU_ts = base_test + test_GRU
    gru_format = 'cart'
elif abs(HR_fren_overall - GRU_REF_OVERALL) < 0.005:
    pred_GRU_tr = pred_fren
    test_GRU_inv = (test_GRU[:, 0:1] * eL_ts + test_GRU[:, 1:2] * eN_ts + test_GRU[:, 2:3] * eB_ts)
    pred_GRU_ts = base_test + test_GRU_inv
    gru_format = 'frenet'
else:
    raise RuntimeError('GRU cache 형태 불명 — EXP #34 진행 불가')

err_GRU = np.linalg.norm(y_train - pred_GRU_tr, axis=1)
hit_GRU = err_GRU < HIT_RADIUS
HR_GRU = hr_triple(hit_GRU, minority_mask_tr)
print(f'\nGRU form={gru_format}  HR: overall={HR_GRU["overall"]:.4f}  major={HR_GRU["major"]:.4f}  minor={HR_GRU["minor"]:.4f}')

# K15/GRU 2-way oracle ceiling
choose_GRU = err_GRU < err_K15
pred_oracle_KG = np.where(choose_GRU[:, None], pred_GRU_tr, pred_K15_tr)
err_oracle_KG = np.linalg.norm(y_train - pred_oracle_KG, axis=1)
HR_oracle_KG = hr_triple(err_oracle_KG < HIT_RADIUS, minority_mask_tr)
print(f'\nK15/GRU 2-way oracle: overall={HR_oracle_KG["overall"]:.4f}  Δo +{HR_oracle_KG["overall"]-K15_REF_HR["overall"]:.4f}')


## 셀 5 — Phase 0: selector + recovery_mask + #32 classifier 재현 + Soft baseline + **#33 best (T=1.5) 비트 일치**

`y_sel_kg = (err_GRU < err_K15)` — 1 = GRU wins.
**#32 classifier OOF/test 로드 + Soft baseline (T=1.0) + #33 best (T=1.5) HR 비트 일치 sanity** = Phase A 핵심 가드.
T=1.5 재현이 Floor 기준값 (+0.0038) 의 정합성 확인.


In [ ]:
y_sel_kg = (err_GRU < err_K15).astype(np.int8)
n_GRUw = int(y_sel_kg.sum())
n_K15w = int((y_sel_kg == 0).sum())
print(f'GRU wins: {n_GRUw} ({n_GRUw/len(y_sel_kg):.4f}) | K15 wins: {n_K15w} ({n_K15w/len(y_sel_kg):.4f})')

both_kg = hit_K15 & hit_GRU
K15_only_kg = hit_K15 & ~hit_GRU
GRU_only_kg = ~hit_K15 & hit_GRU
neither_kg = ~hit_K15 & ~hit_GRU
print(f'Agreement: both {int(both_kg.sum())} | K15_only {int(K15_only_kg.sum())} | GRU_only {int(GRU_only_kg.sum())} | neither {int(neither_kg.sum())}')

recovery_mask = K15_only_kg | GRU_only_kg
print(f'Recovery mask: {int(recovery_mask.sum())} sample ({recovery_mask.mean():.4f})')

# === #32 classifier OOF/test 로드 ===
from sklearn.metrics import roc_auc_score
oof_clf32_per_seed  = [np.load(p) for p in CLF32_OOF_PATHS]
test_clf32_per_seed = [np.load(p) for p in CLF32_TEST_PATHS]
oof_clf32_mean  = np.mean(oof_clf32_per_seed, axis=0).astype(np.float32)
test_clf32_mean = np.mean(test_clf32_per_seed, axis=0).astype(np.float32)
assert oof_clf32_mean.shape == (10000,) and test_clf32_mean.shape == (10000,)
print(f'\n#32 classifier cache 로드: oof_mean {oof_clf32_mean.shape}, test_mean {test_clf32_mean.shape}')
print(f'  oof_clf32 분포: min={oof_clf32_mean.min():.4f}, median={np.median(oof_clf32_mean):.4f}, max={oof_clf32_mean.max():.4f}')

clf32_overall_auc = float(roc_auc_score(y_sel_kg, oof_clf32_mean))
clf32_recovery_auc = float(roc_auc_score(y_sel_kg[recovery_mask], oof_clf32_mean[recovery_mask]))
print(f'#32 classifier metrics (재현): overall AUC={clf32_overall_auc:.4f}  recovery AUC={clf32_recovery_auc:.4f}  (#32 보고 0.5547)')


def _soft_blend_HR(oof_clf, T, Delta, alpha):
    w = apply_soft_lever(oof_clf, T=T, Delta=Delta, alpha=alpha)[:, None]
    pred = w * pred_GRU_tr + (1 - w) * pred_K15_tr
    return hr_triple(np.linalg.norm(y_train - pred, axis=1) < HIT_RADIUS, minority_mask_tr)


# Soft baseline (T=1.0) 재현
HR_soft_base = _soft_blend_HR(oof_clf32_mean, *BASELINE_CELL)
do_base = HR_soft_base['overall'] - K15_REF_HR['overall']
print(f'\nSoft baseline (T=1.0) 재현 HR: overall={HR_soft_base["overall"]:.4f}  Δo=+{do_base:.4f}  (vs #32 보고 +{SOFT_BASELINE_DO:.4f})')
assert abs(HR_soft_base['overall'] - SOFT_BASELINE_HR_OVERALL) < 0.0005,     f'Soft baseline 재현 fail: {HR_soft_base["overall"]:.4f} vs ref {SOFT_BASELINE_HR_OVERALL:.4f}'

# #33 best (T=1.5) 재현 — Floor 기준값 정합성
HR_e33_best = _soft_blend_HR(oof_clf32_mean, *E33_BEST_CELL)
do_e33 = HR_e33_best['overall'] - K15_REF_HR['overall']
print(f'#33 best (T=1.5) 재현 HR: overall={HR_e33_best["overall"]:.4f}  Δo=+{do_e33:.4f}  (vs #33 보고 +{E33_BEST_DO:.4f})')
assert abs(HR_e33_best['overall'] - E33_BEST_HR_OVERALL) < 0.0005,     f'#33 best 재현 fail: {HR_e33_best["overall"]:.4f} vs ref {E33_BEST_HR_OVERALL:.4f}'
print('Soft baseline + #33 best 비트 일치 OK — #32 cache 정합 확인 → Phase A 진입 가능')


## 셀 6 — Phase A: 9 T-point 1-D sweep + per-cell HR + gates

T temperature 단일축. Δ=∞, α=1.0 고정. 학습 0, OOF HR 계산만.

**가드 framework**:
- G1 (informational): √9·0.0014 ≈ +0.0042 (multi-comparison 부담 표시만)
- **Floor (primary)**: Δo > +0.0038 (= #33 best T=1.5 초과)
- **4th guard (primary)**: Δo > +0.0020 (per-cell LB자격)
- G2/G3: Δmajor > −0.002, Δminor > −0.005
- **채택 식**: Floor strict AND 4th guard strict AND G2 AND G3 → OOF Δo max 1개


In [ ]:
def evaluate_cell_gates(HR, floor_thr=FLOOR_PRIMARY):
    d_o = HR['overall'] - K15_REF_HR['overall']
    d_M = HR['major']   - K15_REF_HR['major']
    d_m = HR['minor']   - K15_REF_HR['minor']
    g1_info  = d_o > G1_INFO_THR
    g2       = d_M > -0.002
    g3       = d_m > -0.005
    g4       = d_o > G4_THR
    floor_ok = d_o > floor_thr
    lb_ok    = bool(g4 and g2 and g3 and floor_ok)
    return dict(d_overall=float(d_o), d_major=float(d_M), d_minor=float(d_m),
                G1_info=bool(g1_info), G2=bool(g2), G3=bool(g3),
                G4=bool(g4), Floor=bool(floor_ok), lb_eligible=lb_ok)


print(f'=== Phase A: 9 T-point 1-D sweep (Δ=∞, α=1.0) ===\n')
cell_results = []
_t_pA = time.time()

for T in T_GRID:
    w_cell = apply_soft_lever(oof_clf32_mean, T=T, Delta=FIXED_DELTA, alpha=FIXED_ALPHA)[:, None]
    pred_cell = w_cell * pred_GRU_tr + (1 - w_cell) * pred_K15_tr
    err_cell = np.linalg.norm(y_train - pred_cell, axis=1)
    HR_cell = hr_triple(err_cell < HIT_RADIUS, minority_mask_tr)
    gates_cell = evaluate_cell_gates(HR_cell)
    cell_results.append(dict(
        T=T, Delta=FIXED_DELTA, alpha=FIXED_ALPHA,
        name=cell_name(T, FIXED_DELTA, FIXED_ALPHA),
        is_baseline=(T, FIXED_DELTA, FIXED_ALPHA) == BASELINE_CELL,
        is_e33_best=(T, FIXED_DELTA, FIXED_ALPHA) == E33_BEST_CELL,
        HR=HR_cell, **gates_cell,
        w_range=(float(w_cell.min()), float(w_cell.max())),
    ))
    # 인큐번트(T=1.5, #33 채택) 재선택 금지 — Floor strict float 드리프트 방어 + 동일 제출 슬롯 낭비 차단
    if cell_results[-1]['is_e33_best']:
        cell_results[-1]['lb_eligible'] = False

print(f'Phase A 계산 시간: {time.time()-_t_pA:.2f}s ({N_CELLS} cell)\n')

print(f'{"#":>3} {"name":<18} {"T":>4} {"overall":>8} {"major":>8} {"minor":>8} {"Δo":>8} {"ΔM":>8} {"Δm":>8} {"G1i":>4} {"G2":>3} {"G3":>3} {"G4":>3} {"F":>3} {"LB":>3} {"tag":>10}')
print('-' * 120)
for i, c in enumerate(cell_results):
    tag = '#32 base' if c['is_baseline'] else ('#33 best' if c['is_e33_best'] else '')
    print(f'[{i+1:2d}] {c["name"]:<18} {c["T"]:>4.1f} '
          f'{c["HR"]["overall"]:>8.4f} {c["HR"]["major"]:>8.4f} {c["HR"]["minor"]:>8.4f} '
          f'{c["d_overall"]:>+8.4f} {c["d_major"]:>+8.4f} {c["d_minor"]:>+8.4f} '
          f'{"O" if c["G1_info"] else "X":>4} {"O" if c["G2"] else "X":>3} '
          f'{"O" if c["G3"] else "X":>3} {"O" if c["G4"] else "X":>3} '
          f'{"O" if c["Floor"] else "X":>3} {"O" if c["lb_eligible"] else "X":>3} {tag:>10}')

floor_pass_count = sum(1 for c in cell_results if c['Floor'])
lb_eligible_count = sum(1 for c in cell_results if c['lb_eligible'])
print(f'\nFloor (Δo > +{FLOOR_PRIMARY:.4f} = #33 best 초과) 통과 cell: {floor_pass_count}/{N_CELLS}')
print(f'LB eligible (Floor + G4 + G2 + G3 strict) cell: {lb_eligible_count}/{N_CELLS}')
print(f'G1 informational (Δo > +{G1_INFO_THR:.4f}) 통과 cell: {sum(1 for c in cell_results if c["G1_info"])}/{N_CELLS}')


## 셀 7 — Phase A 분석: T 곡선 (peak / monotonic / plateau 식별) + rank

1-D sweep 이므로 interaction heatmap 불필요. T 곡선 형태로 sub-lever space ceiling 판정.
- **peak interior** (T*∈(0.5,3.0) 내부 최대): T축 local maximum 정밀화 성공
- **monotonic↑** (T=3.0 최대): T 상한 미도달, grid 확장 후보
- **plateau** (Δo 폭 < combined_std): sub-noise zone, T축 ceiling 도달


In [ ]:
do_curve = [c['d_overall'] for c in cell_results]
dm_curve = [c['d_minor'] for c in cell_results]

print('=== T 곡선 (Δo, Δm vs K15) ===\n')
print(f'{"T":>5} {"Δo":>9} {"Δm":>9}  bar(Δo)')
for T, do, dm in zip(T_GRID, do_curve, dm_curve):
    nb_ = int(round((do - min(do_curve)) / (max(do_curve) - min(do_curve) + 1e-9) * 30))
    print(f'{T:>5.1f} {do:>+9.4f} {dm:>+9.4f}  {"#"*nb_}')

i_peak = int(np.argmax(do_curve))
T_peak = T_GRID[i_peak]
do_span = max(do_curve) - min(do_curve)
print(f'\npeak T = {T_peak}  (Δo {do_curve[i_peak]:+.4f})')
print(f'Δo span (max−min) = {do_span:.4f}  (combined_std {COMBINED_STD:.4f})')

# 곡선 형태 판정
if do_span < COMBINED_STD:
    shape = 'plateau (Δo 폭 < combined_std → sub-noise zone, T축 ceiling 도달)'
elif i_peak == len(T_GRID) - 1:
    shape = 'monotonic↑ (T=3.0 최대 → T 상한 미도달, grid 확장 후보)'
elif i_peak == 0:
    shape = 'monotonic↓ (T=0.5 최대 → sharpen 방향, prior 반전 — 재검토)'
else:
    shape = f'peak interior (T*={T_peak} 내부 최대 → T축 local maximum 정밀화)'
print(f'곡선 형태: {shape}')

# vs #33 best (T=1.5) 비교
i_e33 = T_GRID.index(1.5)
print(f'\nvs #33 best (T=1.5, Δo {do_curve[i_e33]:+.4f}):')
for i, T in enumerate(T_GRID):
    if T == 1.5:
        continue
    marg = do_curve[i] - do_curve[i_e33]
    star = ' ★strict 초과' if marg > 0 else ''
    print(f'  T={T:>4.1f}: Δo {do_curve[i]:+.4f}  (vs T=1.5: {marg:+.4f}){star}')

# rank
order = sorted(range(len(T_GRID)), key=lambda i: -do_curve[i])
print(f'\nTop-3 T (OOF Δo): ' + ', '.join(f'T={T_GRID[i]}({do_curve[i]:+.4f})' for i in order[:3]))
print(f'Bot-3 T (OOF Δo): ' + ', '.join(f'T={T_GRID[i]}({do_curve[i]:+.4f})' for i in order[-3:]))


## 셀 8 — Phase 3: best cell 선택 + recall analysis

**채택 식**: Floor strict AND 4th guard strict AND G2 AND G3 통과 cell 중 OOF Δo max 1개.
LB eligible 0개 (모든 T ≤ #33 best +0.0038) → best_cell None → LB 0.6914 유지.


In [ ]:
phase_a_eligible = [c for c in cell_results if c['lb_eligible']]

print(f'=== Phase 3: best cell 선택 ===')
print(f'Phase A LB eligible: {len(phase_a_eligible)} cell')

if not phase_a_eligible:
    print(f'\n⚠ LB eligible cell 0개 — LB 슬롯 사용 권장 안 함')
    print(f'  Floor +{FLOOR_PRIMARY:.4f} (= #33 best 초과) strict + G4 +{G4_THR:.4f} strict + G2 + G3 통과 cell 없음')
    print(f'  → T축 sub-lever space ceiling 확정 (#33 T=1.5 = local max). 다음 lever = L19 (Classifier 강화)')
    best_cell = None
else:
    phase_a_eligible.sort(key=lambda x: -x['d_overall'])
    best_cell = phase_a_eligible[0]
    print(f'\nbest cell ({len(phase_a_eligible)} eligible 중 OOF Δo max):')
    print(f'  name: {best_cell["name"]}  (T={best_cell["T"]})')
    print(f'  Δo: {best_cell["d_overall"]:+.4f}  ΔM: {best_cell["d_major"]:+.4f}  Δm: {best_cell["d_minor"]:+.4f}')
    print(f'  vs #33 best (T=1.5) margin: {best_cell["d_overall"] - E33_BEST_DO:+.4f} (OOF)')

# Recall analysis
print(f'\n=== Recall analysis ===')
print(f'K15/GRU recovery: K15_only {int(K15_only_kg.sum())} / GRU_only {int(GRU_only_kg.sum())}')


def _recall_cell(w_arr):
    use_GRU = w_arr > 0.5
    GRU_only_recovered = int((use_GRU & GRU_only_kg).sum())
    K15_only_lost = int((use_GRU & K15_only_kg).sum())
    return dict(GRU_selected=int(use_GRU.sum()),
                GRU_only_recovered=GRU_only_recovered,
                K15_only_lost=K15_only_lost,
                net_gain_hit=GRU_only_recovered - K15_only_lost)


print(f'\n#33 best (T=1.5):')
print(f'  {_recall_cell(apply_soft_lever(oof_clf32_mean, *E33_BEST_CELL))}')
if best_cell is not None:
    w_best = apply_soft_lever(oof_clf32_mean, T=best_cell['T'], Delta=best_cell['Delta'], alpha=best_cell['alpha'])
    print(f'\nBest cell ({best_cell["name"]}):')
    print(f'  {_recall_cell(w_best)}')


## 셀 9 — Submission 생성 (best 1개만, memory feedback-submission-best-only)

**plan §4.1 + memory `feedback-submission-best-only`**: Phase 3 선정 best cell **1개만** 생성. 9 T-point HR/가드 정량 자산은 `results/t_grid_e34_meta.json` 에 전부 보존.

best_cell None (LB eligible 0) 시 submission 0개 → #33 LB 0.6914 유지.

명명: `submission_split_e34_{best_cell_name}_ms3.csv` (예: `T18_DINF_a100` = T=1.8).


In [ ]:
def make_submission(pred_test_abs, name):
    assert pred_test_abs.shape == (10000, 3)
    assert np.isfinite(pred_test_abs).all()
    df = pd.DataFrame({
        'id': test_ids,
        'x': pred_test_abs[:, 0],
        'y': pred_test_abs[:, 1],
        'z': pred_test_abs[:, 2],
    })
    df.to_csv(name, index=False)
    return name


print('=== Submission 생성 (best 1개만) ===\n')
sub_paths = {}

if best_cell is None:
    print(f'best_cell None (LB eligible cell 0개)')
    print(f'→ submission 0개 생성. LB 슬롯 사용 권장 안 함.')
    print(f'→ #33 K15/GRU Soft + T=1.5 LB 0.6914 유지가 최종.')
    chosen_lb_path = None
else:
    w_test_best = apply_soft_lever(
        test_clf32_mean,
        T=best_cell['T'], Delta=best_cell['Delta'], alpha=best_cell['alpha'],
    )[:, None]
    pred_test_best = w_test_best * pred_GRU_ts + (1 - w_test_best) * pred_K15_ts
    chosen_lb_path = f'submission_split_e34_{best_cell["name"]}_ms3.csv'
    make_submission(pred_test_best, chosen_lb_path)
    sub_paths[best_cell['name']] = chosen_lb_path
    print(f'  best cell: {best_cell["name"]}  (T={best_cell["T"]})')
    print(f'  Δo OOF: {best_cell["d_overall"]:+.4f}  margin vs #33 best: {best_cell["d_overall"]-E33_BEST_DO:+.4f}')
    print(f'  → submission: {chosen_lb_path}')
    print(f'  → LB 슬롯 1개 권장 (LB > #33 0.6914 + 0.0001 margin 확인)')


## 셀 10 — Meta 저장 + Drive 복사 + files.download (best 1개만, memory feedback-submission-best-only)

In [ ]:
def safe(x):
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, np.ndarray): return x.tolist()
    if x == float('inf'): return 'inf'
    return x


def _cell_to_dict(c):
    d = dict(c)
    if d.get('Delta') == float('inf'):
        d['Delta'] = 'inf'
    return d


meta = {
    'protocol': 'EXP #34 L18 T grid 정교화 (sub-lever T 단일축 9-point fine sweep, Δ=∞ α=1.0 고정)',
    'seeds': SEEDS, 'n_folds': N_FOLDS,
    'minority_threshold': MINORITY_THRESHOLD,
    'baseline_cell': dict(T=BASELINE_CELL[0], Delta='inf', alpha=BASELINE_CELL[2]),
    'e33_best_cell': dict(T=E33_BEST_CELL[0], Delta='inf', alpha=E33_BEST_CELL[2]),
    'grid': {'sigma_T': T_GRID, 'fixed_Delta': 'inf', 'fixed_alpha': FIXED_ALPHA, 'n_cells': N_CELLS},
    'thresholds': {
        'g1_informational': G1_INFO_THR,
        'g4_primary_per_cell': G4_THR,
        'floor_primary_e33_best': FLOOR_PRIMARY,
        'soft_baseline_do_ref': SOFT_BASELINE_DO,
        'e33_best_do_ref': E33_BEST_DO,
        'combined_std': COMBINED_STD,
    },
    'k15_ref_HR': K15_REF_HR,
    'gru_format': gru_format,
    'gru_HR': HR_GRU,
    'oracle_HR_K15GRU': HR_oracle_KG,
    'ceiling_gain_K15GRU': CEILING_GAIN_KG,
    'agreement_K15_GRU': dict(both=int(both_kg.sum()), K15_only=int(K15_only_kg.sum()),
                                GRU_only=int(GRU_only_kg.sum()), neither=int(neither_kg.sum())),
    'recovery_mask_count': int(recovery_mask.sum()),
    'soft_baseline_reproduce_HR': HR_soft_base,
    'e33_best_reproduce_HR': HR_e33_best,
    'clf32_metrics_reproduce': dict(overall_auc=clf32_overall_auc, recovery_auc=clf32_recovery_auc),
    'phase_a': {
        'n_cells': N_CELLS,
        'cell_results': [_cell_to_dict(c) for c in cell_results],
        'do_curve': do_curve,
        'dm_curve': dm_curve,
        'peak_T': T_peak,
        'do_span': do_span,
        'curve_shape': shape,
        'floor_pass_count': floor_pass_count,
        'lb_eligible_count': lb_eligible_count,
    },
    'best_cell': _cell_to_dict(best_cell) if best_cell is not None else None,
    'lb_chosen_path': chosen_lb_path,
    'submission_best_only': sub_paths,
    'submission_policy': 'best_only (memory feedback-submission-best-only)',
    'minority_train_count': int(minority_mask_tr.sum()),
    'minority_test_count': int(minority_mask_ts.sum()),
}
with open('results/t_grid_e34_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=safe)
print('results/t_grid_e34_meta.json 저장 (9 T-point HR + 가드 + T 곡선 정량 자산 전부 보존)')

# Drive 복사 + files.download (best 1개만)
try:
    from google.colab import drive, files
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive'
    results_drive = os.path.join(DRIVE_BASE, 'results')
    os.makedirs(results_drive, exist_ok=True)
    !cp -r results/* {results_drive}/
    if os.path.exists('hr_aware_t_grid_e34.ipynb'):
        !cp hr_aware_t_grid_e34.ipynb {DRIVE_BASE}/
    if sub_paths:
        for p in sub_paths.values():
            !cp {p} {DRIVE_BASE}/
            files.download(p)
        print(f'Drive 복사 + 다운로드 완료 (best submission 1개: {list(sub_paths.values())[0]})')
    else:
        print('best_cell None → submission 0개. Drive 복사 skip (meta.json + ipynb만 복사)')
except ImportError:
    print('Colab 아님 — Drive 복사 skip')
